# 06 — Live ingestion: one flight, its alternatives, and where its aircraft is

<!-- contract -->

| | |
|---|---|
| **Reads** | AeroDataBox (schedules, gate times), OpenSky (live state) |
| **Writes** | `api_bronze_flights`, `api_silver_flights`, `opensky_states` |
| **Runtime** | ~2 min |
| **Requires** | Secrets `aerodatabox_key`, `opensky_client_id`, `opensky_client_secret` |

Two inputs. A flight number and a date. Everything else is derived.

## Why the inputs shrank

The previous version asked for a flight number, an origin, a destination and a carrier code,
and got them wrong in ways that were hard to see: one run passed `airline_iata="DL1682"`,
which returns nothing because that field wants a carrier code rather than a flight number.
Four inputs that must agree with each other is four chances to be wrong, and the failure
looks identical to "no flights today".

`AeroDataBox`'s `/flights/number/{number}/{date}` derives the itinerary from the number
alone — `DL1572` comes back as `ATL -> IAH` with scheduled and revised times, distance,
aircraft, and airline. Origin and destination are *outputs*. So they stopped being inputs.

## Why AeroDataBox and not AviationStack

The AviationStack plan in use returns schedules dated roughly two weeks in the past. That
does not merely degrade the live path, it makes it impossible: a flight from a fortnight ago
cannot be airborne now, so every attempt to match one against a live ADS-B snapshot returned
zero by construction. Every `match_rate 0.0%` in the earlier runs had that single cause.

AeroDataBox returns the flight operating on the requested date, and carries two identifiers
that matter more than the schedule itself:

| Field | Value | Why |
|---|---|---|
| `callSign` | `DAL1572` | OpenSky's callsign, supplied directly — no IATA-to-ICAO table |
| `aircraft.modeS` | `A34729` | OpenSky's `icao24` — a **unique airframe**, not a flight number |

Matching on the airframe address is the strongest join between the two feeds. Codeshares
share a flight number but not an aeroplane, and a flight number recurs daily while an
airframe address does not.

## The division of labour, unchanged

AeroDataBox answers **how late** on gate semantics — `revisedTime - scheduledTime` is the
same quantity BTS records as `DEP_DELAY` and the models were trained on. OpenSky answers
**where and what phase**. Deriving delay from ADS-B would give wheels-off, which differs
from gate delay by taxi-out and is worst at the congested airports where delay matters most.


In [0]:
%pip install holidays -q
dbutils.library.restartPython()


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import sys
sys.path.append("..")

from datetime import date, datetime, time, timedelta, timezone

import pandas as pd
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.functions import current_timestamp, lit

from src import config
from src.aerodatabox import (
    AeroDataBoxClient, departure_to_row, flight_to_row, normalise_flight_number,
)
from src.faa_nas import fetch_status, summarise
from src.opensky import (
    derive_schedule, fetch_departures,
    CONUS_BBOX, OpenSkyClient, match_by_airframe, match_to_schedule, parse_states,
    split_by_phase,
)


## The inputs

`FLIGHT_NUMBER` is forgiving: `dl1572`, `DL 1572` and `DL-1572` all normalise to `DL1572`,
because a person typing a flight number should not have to get the spacing right.

`FLIGHT_DATE` matters more than it looks. The provider serves a window around today - a few
days back and a few days forward - so the date chooses which question is being asked. A past
date gives a flight that has already operated, with its actual times filled in. Today gives
the live picture, and it is the only choice where the OpenSky match can succeed, because
that is the only day an aircraft is where the snapshot can see it. A future date gives the
published schedule with no revisions yet, which is the pure pre-departure case.

`ORIGIN` and `DESTINATION` are **optional tie-breakers, not lookups**. The route is derived
from the flight number - that is the point of the design, and it costs no extra call, because
origin and destination come back in the same response that was needed anyway. They exist
because one flight number can operate more than one leg in a day: DL1572 might fly ATL->IAH
in the morning and IAH->ATL in the afternoon, and "DL1572 today" does not say which one a
traveller means. Filling these in says it. Leave them blank and the notebook picks the next
leg due to depart, which is the one a traveller usually means but not the one they always
mean.

In [0]:
dbutils.widgets.text("FLIGHT_NUMBER", "DL1572")
dbutils.widgets.text("FLIGHT_DATE", date.today().isoformat())
# Optional. Blank means "whichever leg is next to depart".
dbutils.widgets.text("ORIGIN", "")
dbutils.widgets.text("DESTINATION", "")
dbutils.widgets.dropdown("FIND_ALTERNATIVES", "true", ["true", "false"])
dbutils.widgets.dropdown("USE_OPENSKY", "true", ["true", "false"])

FLIGHT_NUMBER = normalise_flight_number(dbutils.widgets.get("FLIGHT_NUMBER"))
FLIGHT_DATE = date.fromisoformat(dbutils.widgets.get("FLIGHT_DATE").strip())
WANT_ORIGIN = dbutils.widgets.get("ORIGIN").strip().upper() or None
WANT_DEST = dbutils.widgets.get("DESTINATION").strip().upper() or None
FIND_ALTERNATIVES = dbutils.widgets.get("FIND_ALTERNATIVES") == "true"
USE_OPENSKY = dbutils.widgets.get("USE_OPENSKY") == "true"

offset = (FLIGHT_DATE - date.today()).days
horizon = ("today"
           if offset == 0 else
           f"{abs(offset)} day(s) {'ahead' if offset > 0 else 'behind'}")

print(f"Flight : {FLIGHT_NUMBER}")
print(f"Date   : {FLIGHT_DATE}  ({horizon})")
if WANT_ORIGIN or WANT_DEST:
    print(f"Leg    : {WANT_ORIGIN or '*'} -> {WANT_DEST or '*'}  (you specified this)")
else:
    print("Leg    : whichever is next to depart (set ORIGIN/DESTINATION to pin one)")
if offset != 0:
    print("\n  The OpenSky snapshot is always live. Matching a flight from another")
    print("  day against it cannot succeed, so the phase split will be 'unknown'")
    print("  and everything routes to the pre-departure model. That is correct")
    print("  behaviour, not a failure - set the date to today to exercise the join.")

# One id per ingestion, so "this run" is answerable after the fact and
# is_flight_of_interest cannot leak across runs - yesterday's subject stayed
# flagged and kept reappearing in today's board.
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
print(f"Ingest run: {RUN_ID}")

Flight : DL1131
Date   : 2026-09-18  (today)
Leg    : ATL -> IAH  (you specified this)
Ingest run: 20260918T221514Z


## The flight of interest

In [ ]:
adb = AeroDataBoxClient(
    dbutils.secrets.get(config.AERODATABOX_SECRET_SCOPE, config.AERODATABOX_SECRET_KEY)
)

legs = adb.flight_by_number(FLIGHT_NUMBER, FLIGHT_DATE)
print(f"AeroDataBox returned {len(legs)} leg(s) for {FLIGHT_NUMBER} on {FLIGHT_DATE}")
print(f"Quota: {adb.last_quota.get('units_remaining')}/{adb.last_quota.get('units_limit')} "
      f"units, {adb.last_quota.get('requests_remaining')}/"
      f"{adb.last_quota.get('requests_limit')} requests remaining")

focus_rows = [r for r in (flight_to_row(leg) for leg in legs) if r]
if not focus_rows:
    raise ValueError(
        f"{FLIGHT_NUMBER} has no usable record on {FLIGHT_DATE}. The provider serves a "
        "window around today; try a date within a few days, or check the number."
    )

for r in focus_rows:
    delay = "not yet departed" if r["dep_delay"] is None else f"{r['dep_delay']:+.0f} min"
    print(f"\n  {r['flight_iata']}  {r['origin_airport_code']} -> "
          f"{r['destination_airport_code']}  on {r['flight_date']}")
    # Local clock at each airport — what a passenger reads on a ticket, and what
    # BTS recorded as CRS_DEP_TIME. The UTC instant is printed beside it because
    # the OpenSky snapshot is timestamped in UTC and the two are easy to confuse.
    print(f"    scheduled departure {r['crs_dep_time']:04d} local "
          f"({r['origin_timezone']})    arrival {r['crs_arr_time']:04d} local")
    print(f"    same departure in UTC: {r['scheduled_departure_utc']:%H%MZ}    "
          f"block {r['crs_elapsed_time']:.0f} min    {r['distance']:.0f} km")
    print(f"    gate departure delay: {delay}")
    print(f"    aircraft {r['aircraft_reg']} ({r['aircraft_model']})  "
          f"callsign {r['flight_icao']}  icao24 {r['aircraft_icao24']}")
    print(f"    status {r['flight_status']}   quality {r['data_quality']}")

# Pick the leg.
#
# One call to /flights/number/{number}/{date} returns every leg that number flies
# that day, and the route comes back inside it - deriving the route costs nothing
# extra, it is a field of a response this notebook needed regardless.
#
# What does need deciding is *which* leg. If ORIGIN/DESTINATION were given, they
# decide it: an explicit instruction outranks a heuristic, and a traveller who
# names the route is telling us which flight they are on. If they were not, "next
# to depart" is the leg a traveller usually means - for a flight still ahead of
# them it is the one they are booked on, and once every leg has gone it is the
# most recent, which is the one whose delay is worth knowing. Departure time is
# compared in UTC because the legs sit in different time zones and their local
# clocks are not orderable against each other.
if WANT_ORIGIN or WANT_DEST:
    wanted = [r for r in focus_rows
              if (not WANT_ORIGIN or r["origin_airport_code"] == WANT_ORIGIN)
              and (not WANT_DEST or r["destination_airport_code"] == WANT_DEST)]
    if not wanted:
        available = ", ".join(f"{r['origin_airport_code']}->{r['destination_airport_code']}"
                              for r in focus_rows)
        raise ValueError(
            f"{FLIGHT_NUMBER} on {FLIGHT_DATE} flies no leg matching "
            f"{WANT_ORIGIN or '*'} -> {WANT_DEST or '*'}. It flies: {available}. "
            "Clear the ORIGIN/DESTINATION widgets to take the next leg to depart, "
            "or correct them."
        )
    if len(wanted) < len(focus_rows):
        print(f"\nORIGIN/DESTINATION selected {len(wanted)} of {len(focus_rows)} leg(s).")
    focus_rows = wanted

if len(focus_rows) > 1:
    now = datetime.now(timezone.utc)
    upcoming = [r for r in focus_rows if r["scheduled_departure_utc"] >= now]
    chosen = (min(upcoming, key=lambda r: r["scheduled_departure_utc"]) if upcoming
              else max(focus_rows, key=lambda r: r["scheduled_departure_utc"]))
    dropped = [r for r in focus_rows if r is not chosen]
    focus_rows = [chosen]
    label = "next to depart" if upcoming else "most recent departure"
    print(f"\n{FLIGHT_NUMBER} operates {len(dropped) + 1} legs on {FLIGHT_DATE}. "
          f"Scoring the {label}:")
    print(f"  kept    {chosen['origin_airport_code']} -> "
          f"{chosen['destination_airport_code']} at {chosen['crs_dep_time']:04d} local")
    for r in dropped:
        print(f"  skipped {r['origin_airport_code']} -> "
              f"{r['destination_airport_code']} at {r['crs_dep_time']:04d} local")
    print()
    print("  IMPORTANT if you intend to grade this forecast against the outcome.")
    print("  'Next to depart' is evaluated fresh on every run, so once this leg has")
    print("  gone, a later run picks the one after it instead -- the flight you are")
    print("  waiting on is never re-fetched and its arrival never reaches the table.")
    print(f"  Set ORIGIN={chosen['origin_airport_code']} and "
          f"DESTINATION={chosen['destination_airport_code']} to pin this leg, and")
    print("  use the same values on the run you make after it lands.")

focus = focus_rows[0]
ORIGIN, DESTINATION = focus["origin_airport_code"], focus["destination_airport_code"]
print(f"\nScoring one flight: {FLIGHT_NUMBER}  {ORIGIN} -> {DESTINATION} "
      f"on {focus['flight_date']}, scheduled {focus['crs_dep_time']:04d} local")

## Alternatives on the same route

One call to the origin airport, windowed around the flight's own departure, filtered to the
same destination. Codeshares are excluded at the source: one aircraft sold under three flight
numbers would otherwise appear as three alternatives to itself, which is what the earlier
AviationStack runs produced — `DL753`, `WS6993` and `AM4626` were one aeroplane leaving ATL
at 18:47.


In [0]:
alt_rows = []
if not FIND_ALTERNATIVES:
    print("FIND_ALTERNATIVES=false — scoring the flight of interest alone.")
else:
    dep_local = datetime.combine(
        focus["flight_date"], time(focus["crs_dep_time"] // 100, focus["crs_dep_time"] % 100)
    )
    span = timedelta(hours=config.ALTERNATIVE_SEARCH_HOURS)
    window_start, window_end = dep_local - span, dep_local + span

    departures = adb.airport_departures(ORIGIN, window_start, window_end)
    print(f"{ORIGIN} departures in +/-{config.ALTERNATIVE_SEARCH_HOURS}h: {len(departures)}")
    print(f"Quota: {adb.last_quota.get('units_remaining')} units remaining")

    candidates = [r for r in (departure_to_row(d, ORIGIN) for d in departures) if r]
    same_route = [r for r in candidates if r["destination_airport_code"] == DESTINATION]
    print(f"  of which {ORIGIN} -> {DESTINATION}: {len(same_route)}")

    focus_numbers = {r["flight_iata"] for r in focus_rows}
    alt_rows = [r for r in same_route if r["flight_iata"] not in focus_numbers]
    print(f"  excluding the flight of interest itself: {len(alt_rows)} alternatives")

    if not alt_rows and same_route:
        print("\n  Every same-route departure in the window IS the flight of interest.")
    elif not alt_rows:
        print(f"\n  Nothing else flies {ORIGIN} -> {DESTINATION} in this window. Widen")
        print("  ALTERNATIVE_SEARCH_HOURS in config, or accept that the route is thin.")


ATL departures in +/-4h: 557
Quota: 334 units remaining
  of which ATL -> IAH: 8
  excluding the flight of interest itself: 7 alternatives


## Assemble the pool

In [0]:
rows = [dict(r, is_flight_of_interest=True) for r in focus_rows]
rows += [dict(r, is_flight_of_interest=False) for r in alt_rows]

silver_pdf = pd.DataFrame(rows)

# Same route, same minute, two operating carriers is a codeshare the provider's
# filter missed rather than a genuine choice.
dupe_key = ["origin_airport_code", "destination_airport_code", "flight_date", "crs_dep_time"]
dupes = silver_pdf.duplicated(subset=dupe_key, keep="first").sum()
if dupes:
    silver_pdf = (
        silver_pdf.sort_values("is_flight_of_interest", ascending=False)
        .drop_duplicates(subset=dupe_key, keep="first")
        .copy()
    )
    print(f"Collapsed {dupes} same-minute duplicate(s) on the same route")

HAS_ROWS = len(silver_pdf) > 0
print(f"Pool: {len(silver_pdf)} flights "
      f"({int(silver_pdf['is_flight_of_interest'].sum())} of interest, "
      f"{int((~silver_pdf['is_flight_of_interest']).sum())} alternatives)")

known = silver_pdf["dep_delay"].notna().sum()
print(f"Carrying a gate departure delay: {known} of {len(silver_pdf)}")
print("  Those can use the in-flight model. The rest get pre-departure, which is")
print("  the only variant that works before an aircraft has left.")


Collapsed 1 same-minute duplicate(s) on the same route
Pool: 7 flights (1 of interest, 6 alternatives)
Carrying a gate departure delay: 5 of 7
  Those can use the in-flight model. The rest get pre-departure, which is
  the only variant that works before an aircraft has left.


## Raw payload to Bronze

In [0]:
import json as _json

raw = _json.dumps({"flight": legs, "alternatives_count": len(alt_rows),
                   "flight_number": FLIGHT_NUMBER, "flight_date": FLIGHT_DATE.isoformat()})
(
    spark.createDataFrame([(raw,)], ["raw_payload"])
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source", lit("aerodatabox"))
    .write.format("delta").mode("append").option("mergeSchema", "true")
    .saveAsTable(config.API_BRONZE)
)
print(f"Appended raw payload -> {config.API_BRONZE}")


Appended raw payload -> workspace.flights.api_bronze_flights


## Live aircraft state

One call returns the whole tracked airspace. The match is on `icao24`, the airframe address
AeroDataBox gave us as `aircraft.modeS` — a specific aeroplane rather than a flight number.
Callsign matching runs as a fallback for flights with no tail assigned yet.


In [0]:
state_rows, phase_by_key, opensky_report = [], {}, None

if not USE_OPENSKY:
    print("USE_OPENSKY=false — skipping the live state feed.")
else:
    try:
        opensky = OpenSkyClient(
            dbutils.secrets.get(config.OPENSKY_SECRET_SCOPE, config.OPENSKY_CLIENT_ID_KEY),
            dbutils.secrets.get(config.OPENSKY_SECRET_SCOPE, config.OPENSKY_CLIENT_SECRET_KEY),
        )
        state_rows = parse_states(opensky.fetch_states(CONUS_BBOX))
        ground, airborne = split_by_phase(state_rows)
        print(f"OpenSky: {len(state_rows):,} aircraft in ONE call "
              f"(token refreshes: {opensky.refresh_count})")
        print(f"  on ground {len(ground):,}   airborne {len(airborne):,}")

        airframes = [a for a in silver_pdf.get("aircraft_icao24", []) if a]
        by_frame, frame_report = match_by_airframe(state_rows, airframes)
        print(f"\nAirframe match (icao24): {frame_report['matched']} of "
              f"{frame_report['airframes_wanted']} aircraft located")

        callsigns = [c for c in silver_pdf.get("flight_icao", []) if c]
        by_call, call_report = match_to_schedule(state_rows, callsigns)
        print(f"Callsign match (fallback): {call_report['matched']} of "
              f"{call_report['scheduled_flights']} flights located")

        frame_to_phase = {
            (r.get("icao24") or "").lower():
                ("airborne" if r["on_ground"] is not True else "on_ground")
            for r in by_frame
        }
        call_to_phase = {
            r["callsign"]: ("airborne" if r["on_ground"] is not True else "on_ground")
            for r in by_call
        }
        phase_by_key = {"airframe": frame_to_phase, "callsign": call_to_phase}
        opensky_report = {"airframe": frame_report, "callsign": call_report}

        if not frame_to_phase and not call_to_phase:
            print("\n  Nothing matched. Expected unless the date is today and the")
            print("  aircraft is moving: a parked aeroplane with its transponder off")
            print("  broadcasts nothing, and a flight on another date is not in a live")
            print("  snapshot at all.")
    except Exception as e:
        print(f"OpenSky unavailable ({type(e).__name__}: {e}).")
        print("Continuing — every flight falls back to pre-departure.")


OpenSky: 6,885 aircraft in ONE call (token refreshes: 1)
  on ground 689   airborne 6,196

Airframe match (icao24): 2 of 4 aircraft located
Callsign match (fallback): 2 of 5 flights located


## Context and corroboration, then the write

Three things still have to reach the row before it is durable: the derived schedule, the FAA
airspace conditions, and the phase split computed above. The write is the last cell of the
notebook for that reason - an earlier version wrote `api_silver` two cells before attaching
`flight_phase`, and the column never reached the table at all.

## Airspace conditions — context, not a feature

`02_eda` put **NAS at 19.3% of delay minutes** and 47.7% of delayed flights. NAS is airspace
and air-traffic congestion: ground delay programmes, ground stops, runway construction,
volume. Nothing in the feature set touches it, because nothing in the BTS extract describes
the state of the airspace on the day a flight operated.

The FAA publishes exactly that, live, free, and without a key.

**It is deliberately not a model feature.** There is no historical archive — the endpoint
reports conditions *now* — so the matching column cannot be built for 2019-2023, and a model
that never saw it cannot be scored on it. Feeding it in anyway would be worse than leaving it
out.

So it sits beside the prediction instead. A model saying 16% while Atlanta is under a ground
delay programme averaging 45 minutes is not wrong; it simply never knew. A reader who sees
both is better informed than one who sees either, and labelling which is which is the part
that keeps it honest.


### Corroborating the schedule with what actually happened

AeroDataBox says when the flight is *meant* to leave. OpenSky can say when it *has been*
leaving: a flight number is a recurring daily service, so the median time of day its callsign
has been observed getting airborne is a timetable derived from observation rather than bought.

Two things come out of it, and the second is the one no published schedule contains.

**The median** is a cross-check. A large gap between it and the published departure means the
flight habitually goes late — which is the same signal `schedule_padding` measures
historically, arrived at from the opposite direction.

**The spread** says how *reliable* the service is. A flight that always gets airborne within
twenty minutes of the same time is a different proposition from one that varies by two hours,
and the schedule looks identical for both.

Both are **wheels-off**, and the models are trained on gate delay — the two differ by taxi-out.
So this corroborates and it fills in when the AeroDataBox quota is spent; it does not feed the
model, which would be train/serve skew of exactly the kind this project has already been
bitten by once.


In [0]:
derived = {}
if USE_OPENSKY:
    try:
        # ICAO form: OpenSky keys airports by ICAO, AeroDataBox gave us both.
        origin_icao = (legs[0].get("departure", {}).get("airport", {}) or {}).get("icao")
        if origin_icao:
            history = fetch_departures(opensky, origin_icao, days=5)
            derived = derive_schedule(history)
            print(f"OpenSky: {len(history):,} departures from {origin_icao} over 5 days")
            print(f"  {len(derived):,} callsigns observed at least 3 times")

            entry = derived.get(focus["flight_icao"])
            if entry:
                gap = entry["median_hhmm"] - focus["crs_dep_time"]
                print(f"\n  {focus['flight_icao']} observed {entry['observations']} times")
                print(f"    published departure   {focus['crs_dep_time']:04d} local")
                print(f"    typical wheels-off    {entry['median_hhmm']:04d} UTC "
                      f"(spread {entry['spread_minutes']} min)")
                print(f"    reliability           "
                      f"{'steady' if entry['spread_minutes'] <= 30 else 'variable'}")
                print("\n    Wheels-off, not gate-out, so the two clocks are not directly")
                print("    comparable — taxi-out sits between them. The spread is the part")
                print("    worth reading: it says how consistent this service is, which no")
                print("    published schedule records.")
            else:
                print(f"\n  {focus['flight_icao']} not seen often enough in 5 days to")
                print("  derive a timetable. Expected for a service that does not run daily.")
    except Exception as e:
        print(f"Derived schedule unavailable ({type(e).__name__}: {e})")

_entry = derived.get(focus["flight_icao"]) if derived else None
silver_pdf["observed_departure_hhmm"] = _entry["median_hhmm"] if _entry else None
silver_pdf["observed_spread_minutes"] = _entry["spread_minutes"] if _entry else None


OpenSky: 4,561 departures from KATL over 5 days
  1,030 callsigns observed at least 3 times

  DAL1131 observed 5 times
    published departure   1625 local
    typical wheels-off    2038 UTC (spread 34 min)
    reliability           variable

    Wheels-off, not gate-out, so the two clocks are not directly
    comparable — taxi-out sits between them. The spread is the part
    worth reading: it says how consistent this service is, which no
    published schedule records.


In [0]:
nas_lines, nas_updated = [], None
try:
    nas = fetch_status()
    nas_updated = nas.updated
    nas_lines = summarise(nas, [ORIGIN, DESTINATION])
    print(f"FAA NAS status as of {nas_updated}")
    print(f"  {len(nas.airports_affected)} airports currently reporting conditions")
    if nas_lines:
        print(f"\n  Affecting this route:")
        for line in nas_lines:
            print(f"    {line}")
    else:
        print(f"\n  Neither {ORIGIN} nor {DESTINATION} is reporting a condition.")
except Exception as e:
    # Free, unauthenticated and outside our control. Never fatal.
    print(f"FAA NAS status unavailable ({type(e).__name__}: {e})")

# Carried on the row so 07_score can show it, clearly marked as context that the
# model did not see.
silver_pdf["nas_conditions"] = "; ".join(nas_lines) if nas_lines else None
silver_pdf["nas_checked_at"] = nas_updated


FAA NAS status as of Fri Sep 18 22:15:21 2026 GMT
  10 airports currently reporting conditions

  Neither ATL nor IAH is reporting a condition.


In [0]:
if state_rows:
    (
        spark.createDataFrame(pd.DataFrame(state_rows))
        .withColumn("ingested_at", current_timestamp())
        .write.format("delta").mode("append").option("mergeSchema", "true")
        .saveAsTable(config.OPENSKY_STATES)
    )
    print(f"Appended {len(state_rows):,} state vectors -> {config.OPENSKY_STATES}")

# Airframe first, callsign second, 'unknown' last.
frame_map = (phase_by_key or {}).get("airframe", {})
call_map = (phase_by_key or {}).get("callsign", {})


def _phase(row):
    frame = (row.get("aircraft_icao24") or "").lower()
    if frame and frame in frame_map:
        return frame_map[frame]
    call = row.get("flight_icao")
    if call and call in call_map:
        return call_map[call]
    return "unknown"


silver_pdf["flight_phase"] = [_phase(r) for _, r in silver_pdf.iterrows()]
silver_pdf["ingest_run_id"] = RUN_ID
print(f"Phase: {silver_pdf['flight_phase'].value_counts().to_dict()}")

silver_sdf = (
    spark.createDataFrame(silver_pdf)
    .withColumn("ingested_at", current_timestamp())
)

# MERGE, not append.
#
# A flight is one real thing, and appending made a new row for it on every run:
# 221 rows described 17 flights, and most of them were the same aircraft observed
# repeatedly. Worse, the rows disagreed — an aircraft on the ground at 11:00 and
# airborne at 12:30 produced two rows with different phases and different delays,
# and nothing downstream knew which was current.
#
# The key is the flight itself: number, date, and route. On a match every column
# is overwritten, so the newest observation wins outright — that is what makes
# `on_ground -> airborne` and `dep_delay NULL -> +14 min` a correction rather than
# a second opinion.
MERGE_KEY = ["flight_iata", "flight_date", "origin_airport_code", "destination_airport_code"]

if not spark.catalog.tableExists(config.API_SILVER):
    (silver_sdf.limit(0).write.format("delta").mode("overwrite")
     .option("overwriteSchema", "true").saveAsTable(config.API_SILVER))
    print(f"Created {config.API_SILVER}")
else:
    # A MERGE fails if the source matches one target row more than once, and the
    # two legs of a return trip share a flight number.
    dupes = silver_sdf.count() - silver_sdf.dropDuplicates(MERGE_KEY).count()
    if dupes:
        print(f"Collapsed {dupes} duplicate key(s) within this batch before MERGE")
        silver_sdf = silver_sdf.dropDuplicates(MERGE_KEY)

    # New columns appear as the projection grows; widen the target rather than
    # failing on schema mismatch.
    missing = [f for f in silver_sdf.schema.fields
               if f.name not in set(spark.table(config.API_SILVER).columns)]
    if missing:
        ddl = ", ".join(f"{f.name} {f.dataType.simpleString().upper()}" for f in missing)
        spark.sql(f"ALTER TABLE {config.API_SILVER} ADD COLUMNS ({ddl})")
        print(f"Widened {config.API_SILVER} with {[f.name for f in missing]}")

    # Fix VOID columns - these were created from NULL-only data but now have real values.
    # ALTER COLUMN changes the type so the MERGE can succeed.
    target_schema = {f.name: f.dataType.simpleString() for f in spark.table(config.API_SILVER).schema.fields}
    for f in silver_sdf.schema.fields:
        if f.name in target_schema and target_schema[f.name] == 'void' and f.dataType.simpleString() != 'void':
            spark.sql(f"ALTER TABLE {config.API_SILVER} ALTER COLUMN {f.name} TYPE {f.dataType.simpleString().upper()}")
            print(f"Changed {config.API_SILVER}.{f.name} from VOID to {f.dataType.simpleString().upper()}")

cols = [f.name for f in silver_sdf.schema.fields]
condition = " AND ".join(f"t.{k} = s.{k}" for k in MERGE_KEY)

(
    DeltaTable.forName(spark, config.API_SILVER).alias("t")
    .merge(silver_sdf.alias("s"), condition)
    .whenMatchedUpdate(set={c: F.col(f"s.{c}") for c in cols})
    .whenNotMatchedInsert(values={c: F.col(f"s.{c}") for c in cols})
    .execute()
)

# Exactly one subject at a time.
#
# `is_flight_of_interest` is a property of the *question*, not of the flight, and
# the table outlives the question. MERGE keeps a row per flight across runs, so
# without this the previous subject stayed flagged and 07_score printed two
# "FLIGHT OF INTEREST" blocks with nothing to say which one was asked about.
# Clearing every other row here makes the flag mean "the flight being asked about
# right now", which is what both notebooks read it as.
spark.sql(f"""
    UPDATE {config.API_SILVER}
       SET is_flight_of_interest = false
     WHERE is_flight_of_interest = true
       AND (ingest_run_id IS NULL OR ingest_run_id <> '{RUN_ID}')
""")

still_flagged = (spark.table(config.API_SILVER)
                 .filter(F.col("is_flight_of_interest") == True)  # noqa: E712
                 .count())
print(f"Flights of interest in {config.API_SILVER}: {still_flagged} (expected 1)")

total = spark.table(config.API_SILVER).count()
print(f"MERGE complete. {config.API_SILVER}: {total:,} rows "
      f"({silver_sdf.count()} from this run)")
display(
    spark.table(config.API_SILVER)
    .filter(F.col("ingest_run_id") == RUN_ID)
    .select("is_flight_of_interest", "flight_iata", "origin_airport_code",
            "destination_airport_code", "crs_dep_time", "dep_delay", "flight_phase",
            "aircraft_icao24", "flight_status")
    .orderBy("crs_dep_time")
)

Appended 6,885 state vectors -> workspace.flights.opensky_states
Phase: {'unknown': 5, 'airborne': 2}
Flights of interest in workspace.flights.api_silver_flights: 1 (expected 1)
MERGE complete. workspace.flights.api_silver_flights: 14 rows (7 from this run)


is_flight_of_interest,flight_iata,origin_airport_code,destination_airport_code,crs_dep_time,dep_delay,flight_phase,aircraft_icao24,flight_status
false,DL1572,ATL,IAH,1217,21.0,unknown,a3524e,Departed
false,DL1223,ATL,IAH,1447,10.0,airborne,a3d890,Departed
false,UA3987,ATL,IAH,1603,null,unknown,null,Unknown
true,DL1131,ATL,IAH,1625,14.0,airborne,a3dffe,Arrived
false,UA2376,ATL,IAH,1857,0.0,unknown,a445a4,Expected
false,DL2327,ATL,IAH,1859,0.0,unknown,a3524e,Expected
false,DL1284,ATL,IAH,2017,null,unknown,null,Unknown
